Họ và tên: Phạm Quốc Việt

MSSV: 22119255

Bài tập buổi 4: 8 Puzzles

Link Github: https://github.com/PHAM-QUOC-VIET/Introduction-to-AI-/blob/main/Buoi4/ModelBasedReflexAgent.ipynb

- PEAS:
+ P: Agent đạt được trạng thái đích với số bước di chuyển ít nhất, thời gian tìm kiếm nhỏ và tốn ít bộ nhớ
+ E: Bảng 3x3, 8 ô chứa các số từ 1 tới 8, 1 ô trống (_/0)
+ A: Agent có thể đi lên, xuống, sang trái, sang phải hay dừng thực thi
+ S: Quan sát trạng thái hiện tại của bảng, xác định vị trí ô trống, kiểm tra goal state


- Rules:
+ Nếu trạng thái hiện tại bằng trạng thái đích -> STOP
+ Nếu action làm ô trống đi ra ngoài ma trận -> Agent không được phép đi
+ Nếu trạng thái mới đã tồn tại trong history -> Agent không được phép đi
+ Nếu action tạo trạng thái mới hợp lệ -> Thêm vào valid_actions
+ Nếu trạng thái mới có Manhattan Distance nhỏ hơn -> Ưu tiên chọn action đó
+ Nếu nhiều action hợp lệ tồn tại -> Chọn action có heuristic nhỏ nhất
+ Nếu không còn action hợp lệ -> Agent dừng lại

In [ ]:
import copy

class ModelBasedReflexAgent:
    def __init__(self, start_state, goal_state):
        # Trạng thái nội bộ (Model)
        self.current_state = start_state
        self.goal_state = goal_state
        self.history = [copy.deepcopy(start_state)] # Lưu lịch sử tránh lặp vị trí
        self.moves = {
            'UP': (-1, 0),
            'DOWN': (1, 0),
            'LEFT': (0, -1),
            'RIGHT': (0, 1)
        }

    def find_blank(self, state):
        """Tìm vị trí của ô trống"""
        for r in range(3):
            for c in range(3):
                if state[r][c] == 0:
                    return r, c

    def compute_manhattan(self, state):
        """Hàm Heuristic: Tính tổng khoảng cách Manhattan đến đích"""
        distance = 0
        # Tạo bản đồ vị trí đích để tra cứu nhanh
        goal_pos = {}
        for r in range(3):
            for c in range(3):
                goal_pos[self.goal_state[r][c]] = (r, c)
        
        for r in range(3):
            for c in range(3):
                val = state[r][c]
                if val != 0: # Không tính thực tế cho ô trống
                    g_r, g_c = goal_pos[val]
                    distance += abs(r - g_r) + abs(c - g_c)
        return distance

    def get_valid_actions(self):
        """Condition: Xác định các hành động hợp lệ từ vị trí ô trống"""
        r, c = self.find_blank(self.current_state)
        valid_actions = {}
        
        for action, (dr, dc) in self.moves.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Tạo trạng thái giả định sau khi đi thử
                next_state = copy.deepcopy(self.current_state)
                next_state[r][c], next_state[nr][nc] = next_state[nr][nc], next_state[r][c]
                # Chỉ giữ lại hành động dẫn tới trạng thái chưa từng đi qua
                if next_state not in self.history:
                    valid_actions[action] = next_state
        return valid_actions

    def select_best_action(self, valid_actions):
        """Rule: Chọn hành động tối ưu nhất dựa trên Heuristic thấp nhất"""
        best_action = None
        best_score = float('inf')
        best_state = None

        for action, next_state in valid_actions.items():
            score = self.compute_manhattan(next_state)
            if score < best_score:
                best_score = score
                best_action = action
                best_state = next_state
                
        return best_action, best_state

    def solve(self, max_steps=1000):
        """Vòng lặp thực thi của Agent"""
        print("Trạng thái bắt đầu:")
        self.print_state(self.current_state)
        
        steps = 0
        while self.current_state != self.goal_state and steps < max_steps:
            valid_actions = self.get_valid_actions()
            
            if not valid_actions:
                print("\nAgent bị kẹt vào trạng thái lặp hoặc không có lối thoát (Local Minima)!")
                return False
                
            action, next_state = self.select_best_action(valid_actions)
            
            # Cập nhật Model nội bộ
            self.current_state = next_state
            self.history.append(copy.deepcopy(next_state))
            steps += 1
            
            print(f"\nBước {steps}: Di chuyển ô trống sang [{action}]")
            self.print_state(self.current_state)
            
        if self.current_state == self.goal_state:
            print(f"\nThành công! Đạt trạng thái đích sau {steps} bước.")
            return True
        else:
            print("\nKhông tìm thấy lời giải trong giới hạn bước đi.")
            return False

    def print_state(self, state):
        for row in state:
            print(" ".join(str(x) if x != 0 else "_" for x in row))


if __name__ == "__main__":
    start = [
        [1, 2, 3],
        [4, 0, 6],
        [7, 5, 8]
    ]
    
    goal = [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 0]
    ]

    agent = ModelBasedReflexAgent(start, goal)
    agent.solve()


Trạng thái bắt đầu:
1 2 3
4 _ 6
7 5 8

Bước 1: Di chuyển ô trống sang [DOWN]
1 2 3
4 5 6
7 _ 8

Bước 2: Di chuyển ô trống sang [RIGHT]
1 2 3
4 5 6
7 8 _

Thành công! Đạt trạng thái đích sau 2 bước.


In [11]:
start = [
    [2, 8, 3],
    [1, 6, 4],
    [7, 0, 5]
]

# Trạng thái đích muốn đạt được
goal = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

agent = ModelBasedReflexAgent(start, goal)
agent.solve()


Trạng thái bắt đầu:
2 8 3
1 6 4
7 _ 5

Bước 1: Di chuyển ô trống sang [RIGHT]
2 8 3
1 6 4
7 5 _

Bước 2: Di chuyển ô trống sang [UP]
2 8 3
1 6 _
7 5 4

Bước 3: Di chuyển ô trống sang [LEFT]
2 8 3
1 _ 6
7 5 4

Bước 4: Di chuyển ô trống sang [UP]
2 _ 3
1 8 6
7 5 4

Bước 5: Di chuyển ô trống sang [LEFT]
_ 2 3
1 8 6
7 5 4

Bước 6: Di chuyển ô trống sang [DOWN]
1 2 3
_ 8 6
7 5 4

Bước 7: Di chuyển ô trống sang [DOWN]
1 2 3
7 8 6
_ 5 4

Bước 8: Di chuyển ô trống sang [RIGHT]
1 2 3
7 8 6
5 _ 4

Bước 9: Di chuyển ô trống sang [UP]
1 2 3
7 _ 6
5 8 4

Bước 10: Di chuyển ô trống sang [UP]
1 _ 3
7 2 6
5 8 4

Bước 11: Di chuyển ô trống sang [LEFT]
_ 1 3
7 2 6
5 8 4

Bước 12: Di chuyển ô trống sang [DOWN]
7 1 3
_ 2 6
5 8 4

Bước 13: Di chuyển ô trống sang [DOWN]
7 1 3
5 2 6
_ 8 4

Bước 14: Di chuyển ô trống sang [RIGHT]
7 1 3
5 2 6
8 _ 4

Bước 15: Di chuyển ô trống sang [RIGHT]
7 1 3
5 2 6
8 4 _

Bước 16: Di chuyển ô trống sang [UP]
7 1 3
5 2 _
8 4 6

Bước 17: Di chuyển ô trống sang [UP]
7 1 _
5 2 3

False